# TEKNOFEST KPI — Colab

**Kod = GitHub · Veri = Drive.** Zip yok.

| Ne | Nerede |
|---|---|
| Kod | `/content/teknofest-video-ajan` (`git clone main`) |
| Videolar / gold | `MyDrive/KAIZEN_KPI/data/` |
| Sonuçlar | Drive `exports/` + `predictions_wide/` |

Drive yolu:
```
MyDrive/KAIZEN_KPI/data/videos/{accident,near_miss,normal}/*.mp4
MyDrive/KAIZEN_KPI/data/exports/gold_labels_hepsi.json
```

**Önemli:** `git reset` symlink’leri bozar. Pull/reset sonrası Drive bağlama hücresini tekrar çalıştır.

Runtime → **T4 GPU**, hücreleri sırayla çalıştır.

### 1) GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime → Change runtime type → T4 GPU seç.'

### 2) Drive bağla

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/KAIZEN_KPI')
for sub in ('data/videos/accident', 'data/videos/near_miss', 'data/videos/normal',
            'data/exports', 'data/predictions_wide'):
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print('Drive kökü:', DRIVE_ROOT)

### 3) Repoyu GitHub’dan çek

Kod güncellenince bu hücreyi tekrar çalıştır, sonra **mutlaka 4a** (Drive yeniden bağla).

In [ ]:
import os

REPO = '/content/teknofest-video-ajan'
REPO_URL = 'https://github.com/TulinBabalikKopmaz/KAIZEN_Teknofest26_DilAjanlar-_VideoAnalizKararSistemi.git'
BRANCH = 'main'

if not os.path.exists(REPO):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO}
else:
    %cd {REPO}
    !git fetch --depth 1 origin {BRANCH}
    !git checkout -B {BRANCH} origin/{BRANCH}
    !git reset --hard origin/{BRANCH}

%cd {REPO}
!git log -1 --oneline
!ls scripts/run_kpi_wide.py scripts/colab_kpi_bootstrap.py

### 4a) Drive verisini repo’ya bağla (symlink)

`git reset` sonrası bu hücreyi **her zaman** tekrar çalıştır. Aksi halde `repo video: 0` görürsün (Drive’da video olsa bile).

In [ ]:
%cd /content/teknofest-video-ajan
!python -u scripts/colab_kpi_bootstrap.py \
  --drive-root /content/drive/MyDrive/KAIZEN_KPI \
  --repo /content/teknofest-video-ajan \
  --model qwen2.5vl:7b \
  --skip-ollama

### 4b) Ollama + Qwen indir

5–15 dk sürebilir. `model OK: qwen2.5vl:7b` görünce devam et.

**Not:** Bu hücre `git reset` yapmaz (symlink bozulmasın).

In [ ]:
%cd /content/teknofest-video-ajan
!python -u scripts/colab_kpi_bootstrap.py --only-ollama --model qwen2.5vl:7b
!ollama list

### 5) Veri kontrolü

Symlink kopuksa otomatik yeniden bağlar.

In [ ]:
import shutil
from pathlib import Path

VIDEO_EXTS = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
DRIVE = Path('/content/drive/MyDrive/KAIZEN_KPI/data')
REPO = Path('/content/teknofest-video-ajan/data')

def list_vids(root: Path):
    if not root.exists():
        return []
    return [p for p in root.rglob('*') if p.suffix.lower() in VIDEO_EXTS]

def ensure_link(name: str):
    src = DRIVE / name
    dest = REPO / name
    src.mkdir(parents=True, exist_ok=True)
    if dest.is_symlink() and dest.resolve() == src.resolve():
        print(f'OK symlink: {name}')
        return
    if dest.exists() or dest.is_symlink():
        if dest.is_dir() and not dest.is_symlink():
            shutil.rmtree(dest)
        else:
            dest.unlink()
    dest.symlink_to(src, target_is_directory=True)
    print(f'YENİDEN bağlandı: {dest} → {src}')

for name in ('videos', 'exports', 'predictions_wide', 'frames', 'labels'):
    ensure_link(name)

dv = list_vids(DRIVE / 'videos')
rv = list_vids(REPO / 'videos')
gold = REPO / 'exports' / 'gold_labels_hepsi.json'

print('Drive video:', len(dv))
print('Repo  video:', len(rv), '| symlink=', (REPO / 'videos').is_symlink())
print('gold       :', gold.exists(), gold)

assert gold.exists(), 'gold_labels_hepsi.json eksik (Drive exports/).'
assert len(dv) >= 18, f'Drive\'da {len(dv)} video var; en az 18 lazım.'
assert len(rv) >= 18, (
    f'Repo hâlâ {len(rv)} video görüyor. 4a hücresini tekrar çalıştır.'
)
print('Veri OK — KPI hücresine geç.')

### 6) KPI (18 video)

~30–90 dk. Sonuçlar Drive `exports/` altına yazılır.

In [ ]:
%cd /content/teknofest-video-ajan
!python -u scripts/run_kpi_wide.py \
  --n 18 \
  --seed 42 \
  --model qwen2.5vl:7b \
  --pred-dir data/predictions_wide \
  --no-second-look

print('--- özet ---')
!ls -la data/exports/kpi_wide* 2>/dev/null || ls data/exports/